# 02 — Fine-Tune Tokenizer

Fine-tunes the `KronosTokenizer` on crypto OHLCV data from GCS.
The tokenizer learns a domain-specific codebook for crypto price sequences.

**Prerequisite:** Run `01_data_preparation.ipynb` first.


In [ ]:
import sys, os
sys.path.insert(0, '..')  # repo root

# ── Configuration ───────────────────────────────────────────────
SYMBOL      = 'bnbusdt'
TIMEFRAME   = '1m'
GCS_BUCKET  = 'epochquant-training'
GCS_PROJECT = None

DATA_PATH  = f'gs://{GCS_BUCKET}/processed/{SYMBOL}_{TIMEFRAME}.csv'
SAVE_PATH  = f'../output_models/{SYMBOL}_{TIMEFRAME}'
os.makedirs(SAVE_PATH, exist_ok=True)

print('Data source:', DATA_PATH)


In [ ]:
from training.config import Config

config = Config()
config.dataset_path = DATA_PATH
config.gcs_project  = GCS_PROJECT
config.save_path    = SAVE_PATH
config.epochs       = 3
config.batch_size   = 8

print('Config ready.')
print(f'  dataset_path : {config.dataset_path}')
print(f'  epochs       : {config.epochs}')
print(f'  batch_size   : {config.batch_size}')


In [ ]:
# ── Load data ────────────────────────────────────────────────────
from training.dataset import QlibDataset

train_dataset = QlibDataset('train')
val_dataset   = QlibDataset('val')
print(f'Train samples: {len(train_dataset)}')
print(f'Val samples  : {len(val_dataset)}')


In [ ]:
# ── Load pretrained tokenizer ────────────────────────────────────
import torch
from model.kronos import KronosTokenizer

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = KronosTokenizer.from_pretrained(config.pretrained_tokenizer_path).to(device)
print(f'Tokenizer loaded → {device}')
params = sum(p.numel() for p in tokenizer.parameters())
print(f'Parameters: {params/1e6:.1f}M')


In [ ]:
# ── Run training ─────────────────────────────────────────────────
# NOTE: For multi-GPU, use torchrun from terminal instead:
#   torchrun --standalone --nproc_per_node=NUM_GPUS -m training.train_tokenizer

import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'training.train_tokenizer'],
    cwd='..', capture_output=False
)
print('Exit code:', result.returncode)


In [ ]:
# ── Verify checkpoint was saved ──────────────────────────────────
import os
ckpt_path = os.path.join(SAVE_PATH, 'tokenizer_finetuned', 'checkpoints', 'best_model')
if os.path.exists(ckpt_path):
    files = os.listdir(ckpt_path)
    print(f'Checkpoint saved at: {ckpt_path}')
    print('Files:', files)
else:
    print(f'[WARNING] Checkpoint not found at: {ckpt_path}')
